# Урок 17. Практикум: база данных под задачу

11 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 16](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-16.ipynb) · [Урок 18 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-18.ipynb)

---

Сквозная работа: создать базу, наполнить данными, ответить запросами на содержательные вопросы, оформить выводы.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-17", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Сквозная работа

Сегодня не новая тема, а полный цикл работы с данными: от вопроса
к схеме, от схемы к базе, от базы к ответам. Так выглядит настоящая
работа с данными — и так же устроен любой проект, где данных больше,
чем помещается в голову.

```
1. Какие вопросы должны получить ответ?
2. Какие сущности участвуют и как они связаны?
3. Схема: таблицы, поля, типы, ключи.
4. Создание таблиц и наполнение данными.
5. Запросы, отвечающие на вопросы из пункта 1.
6. Выводы словами.
```

Порядок именно такой. Начинать с таблиц, не зная вопросов, —
обычная ошибка новичка: получается склад данных, из которого ничего
не достать.

### Задача урока

Школьный спортивный клуб просит навести порядок. Есть секции,
ученики и посещения тренировок. Нужно отвечать на вопросы:

1. сколько человек ходит в каждую секцию;
2. какая секция самая посещаемая;
3. кто пропускает больше всех;
4. в каких секциях меньше пяти человек — их предлагают закрыть;
5. есть ли ученики, записанные, но ни разу не пришедшие.

### Схема

```
секции                ученики                     посещения
┌────┬─────────┬────┐ ┌────┬─────────┬──────┬────┐ ┌────┬───────┬──────┬───────┐
│ id │ название│день│ │ id │ фамилия │ класс│секц│ │ id │ученик │ дата │ был   │
└────┴─────────┴────┘ └────┴─────────┴──────┴────┘ └────┴───────┴──────┴───────┘
                             внешний ключ ↑            внешний ключ ↑
```

* ученик записан в одну секцию — связь «один-ко-многим», внешний
  ключ `секция` лежит в таблице учеников;
* посещение относится к ученику и дате, поле `был` — 1 или 0.

Обратите внимание: «был» хранится числом, а не текстом «да»/«нет».
Так его можно суммировать — и `SUM(был)` сразу даст количество
посещений.

### Приёмы, которые понадобятся

| Вопрос | Инструмент |
|---|---|
| сколько в каждой группе | `GROUP BY` + `COUNT` |
| доля, среднее | `AVG`, `ROUND` |
| «меньше пяти» про группу | `HAVING` |
| данные из двух таблиц | `JOIN` |
| «ни одного» | `LEFT JOIN` + `IS NULL` |
| верхушка списка | `ORDER BY … DESC LIMIT` |

## Смотрим, как это работает

### Шаг 1. Создаём базу

In [ ]:
import sqlite3

соединение = sqlite3.connect(":memory:")
курсор = соединение.cursor()

курсор.execute("""CREATE TABLE секции (
    id INTEGER PRIMARY KEY, название TEXT NOT NULL, день TEXT)""")

курсор.execute("""CREATE TABLE ученики (
    id INTEGER PRIMARY KEY, фамилия TEXT NOT NULL, класс TEXT,
    секция INTEGER,
    FOREIGN KEY (секция) REFERENCES секции (id))""")

курсор.execute("""CREATE TABLE посещения (
    id INTEGER PRIMARY KEY, ученик INTEGER, дата TEXT, был INTEGER,
    FOREIGN KEY (ученик) REFERENCES ученики (id))""")

print("Схема создана")

### Шаг 2. Наполняем данными

In [ ]:
курсор.executemany("INSERT INTO секции VALUES (?, ?, ?)", [
    (1, "волейбол", "понедельник"),
    (2, "шахматы", "среда"),
    (3, "лёгкая атлетика", "пятница"),
    (4, "настольный теннис", "вторник"),
])

курсор.executemany("INSERT INTO ученики VALUES (?, ?, ?, ?)", [
    (1, "Иванов", "11А", 1),
    (2, "Петрова", "10Б", 1),
    (3, "Сидоров", "11А", 2),
    (4, "Кузнецова", "10Б", 2),
    (5, "Морозов", "9В", 1),
    (6, "Егорова", "9В", 3),
    (7, "Волков", "11Б", 1),
    (8, "Зайцева", "10А", 2),
    (9, "Орлов", "9А", 4),
])

посещения = []
номер = 1
данные = {
    1: [1, 1, 1, 1],        # Иванов ходит исправно
    2: [1, 0, 1, 1],
    3: [1, 1, 0, 0],
    4: [0, 0, 1, 0],        # Кузнецова почти не ходит
    5: [1, 1, 1, 0],
    6: [1, 1, 1, 1],
    7: [0, 1, 0, 1],
    8: [1, 0, 0, 0],
    # Орлов (9) записан, но ни разу не приходил — записей нет вовсе
}
даты = ["2026-11-03", "2026-11-10", "2026-11-17", "2026-11-24"]

for ученик, отметки in данные.items():
    for дата, был in zip(даты, отметки):
        посещения.append((номер, ученик, дата, был))
        номер += 1

курсор.executemany("INSERT INTO посещения VALUES (?, ?, ?, ?)", посещения)
соединение.commit()

print("Учеников:", курсор.execute("SELECT COUNT(*) FROM ученики").fetchone()[0])
print("Записей о посещениях:", len(посещения))


def показать(запрос):
    строки = курсор.execute(запрос).fetchall()
    заголовки = [описание[0] for описание in курсор.description]
    print(" | ".join(заголовки))
    print("-" * 55)
    for строка in строки:
        print(" | ".join(str(значение) for значение in строка))
    print(f"[строк: {len(строки)}]\n")

Данные о посещениях собраны программой из компактной таблички
«ученик → отметки». Так удобнее: сами данные видно целиком,
а девяносто строк `INSERT` писать не надо.

### Вопрос 1. Сколько человек в каждой секции

In [ ]:
показать("""
SELECT секции.название, COUNT(ученики.id) AS человек
FROM секции
LEFT JOIN ученики ON ученики.секция = секции.id
GROUP BY секции.название
ORDER BY человек DESC, секции.название
""")

`LEFT JOIN` здесь на всякий случай: если появится секция без единого
ученика, она всё равно попадёт в отчёт с нулём.

### Вопрос 2. Самая посещаемая секция

Считаем не записанных, а реально пришедших: `SUM(был)`.

In [ ]:
показать("""
SELECT секции.название,
       SUM(посещения.был) AS пришли,
       COUNT(посещения.id) AS всего_занятий,
       ROUND(100.0 * SUM(посещения.был) / COUNT(посещения.id), 1) AS процент
FROM посещения
JOIN ученики ON посещения.ученик = ученики.id
JOIN секции  ON ученики.секция = секции.id
GROUP BY секции.название
ORDER BY процент DESC
""")

Умножение на `100.0`, а не на `100`, — важная мелочь: при делении
целых чисел SQLite отбросит дробную часть, и все проценты станут
нулями.

### Вопрос 3. Кто пропускает больше всех

In [ ]:
показать("""
SELECT ученики.фамилия, ученики.класс,
       COUNT(посещения.id) - SUM(посещения.был) AS пропусков
FROM посещения
JOIN ученики ON посещения.ученик = ученики.id
GROUP BY ученики.фамилия, ученики.класс
HAVING пропусков > 0
ORDER BY пропусков DESC, ученики.фамилия
""")

### Вопрос 4. Секции, которые предлагают закрыть

In [ ]:
показать("""
SELECT секции.название, COUNT(ученики.id) AS человек
FROM секции
LEFT JOIN ученики ON ученики.секция = секции.id
GROUP BY секции.название
HAVING человек < 5
ORDER BY человек, секции.название
""")

### Вопрос 5. Записан, но ни разу не пришёл

In [ ]:
показать("""
SELECT ученики.фамилия, ученики.класс
FROM ученики
LEFT JOIN посещения ON посещения.ученик = ученики.id
WHERE посещения.id IS NULL
ORDER BY ученики.фамилия
""")

Классический приём «найти тех, у кого ничего нет»: соединяем
`LEFT JOIN` и оставляем строки, где пара не нашлась.

Обратите внимание на разницу с вопросом 3: там мы искали тех, кто
приходил не всегда, здесь — тех, о ком вообще нет записей. Это разные
вопросы и разные запросы, хотя по-русски оба звучат как «кто
не ходит».

### Выводы

По этим пяти запросам можно писать записку директору: волейбол
держится, шахматы посещаются хуже всех, теннис существует
на бумаге — там один записанный, и тот ни разу не пришёл.
Данные не принимают решений, но делают разговор предметным.

## Пробуем сами

Работаем с базой спортивного клуба. Каждая задача — функция без
аргументов, возвращающая `fetchall()`.

### Задача 1. Ученики одной секции

Верните фамилии учеников волейбольной секции по алфавиту.

In [ ]:
def волейболисты():
    return ...

In [ ]:
si.check("1", волейболисты, [
    ((), [("Волков",), ("Иванов",), ("Морозов",), ("Петрова",)]),
])

### Задача 2. Посещений у каждого

Верните «фамилия, количество посещений» (только реальные приходы)
по убыванию, при равенстве — по фамилии. Ученики без записей
в отчёт не попадают.

In [ ]:
def посещений_у_каждого():
    return ...

In [ ]:
si.check("2", посещений_у_каждого, [
    ((), [("Егорова", 4), ("Иванов", 4), ("Морозов", 3), ("Петрова", 3),
          ("Волков", 2), ("Сидоров", 2), ("Зайцева", 1), ("Кузнецова", 1)]),
])

### Задача 3. Секции по дням

Верните «день, количество секций в этот день», упорядоченные
по названию дня.

In [ ]:
def секций_по_дням():
    return ...

In [ ]:
si.check("3", секций_по_дням, [
    ((), [("вторник", 1), ("понедельник", 1), ("пятница", 1), ("среда", 1)]),
])

### Задача 4. Дисциплинированные

Верните фамилии тех, кто не пропустил ни одного занятия
(были на всех четырёх), по алфавиту.

In [ ]:
def без_пропусков():
    return ...

In [ ]:
si.check("4", без_пропусков, [
    ((), [("Егорова",), ("Иванов",)]),
])

### Задача 5. Посещаемость по классам

Верните «класс, сколько раз ученики этого класса приходили»
по убыванию, при равенстве — по названию класса. Классы без записей
не показывать.

In [ ]:
def по_классам():
    return ...

In [ ]:
si.check("5", по_классам, [
    ((), [("9В", 7), ("11А", 6), ("10Б", 4), ("11Б", 2), ("10А", 1)]),
])

### Задача 6. Пустая секция

Верните названия секций, в которых меньше двух записанных учеников,
по алфавиту.

In [ ]:
def маленькие_секции():
    return ...

In [ ]:
si.check("6", маленькие_секции, [
    ((), [("лёгкая атлетика",), ("настольный теннис",)]),
])

### Задача 7. Свои данные в базу

Функция получает список кортежей `(id, фамилия, класс, секция)`,
добавляет их в таблицу `ученики` и возвращает общее количество
учеников в таблице после добавления. Записи с уже занятым `id`
добавлять не нужно — пропускайте их.

In [ ]:
def добавить_учеников(новые):
    return ...

In [ ]:
si.check("7", добавить_учеников, [
    ([(10, "Новиков", "9А", 4)], 10),
    ([(10, "Дубликат", "9А", 4)], 10),
    ([(11, "Ещё", "9Б", 3), (12, "И ещё", "9Б", 3)], 12),
])

## Домашнее задание

### Домашнее задание 1. Отчёт по секциям

Верните «название секции, сколько записано, сколько всего приходов»
для всех секций, включая пустые. По названию секции. Если приходов
нет, должен быть 0.

In [ ]:
def отчёт_по_секциям():
    return ...

In [ ]:
si.check("дз1", отчёт_по_секциям, [
    ((), [("волейбол", 4, 12), ("лёгкая атлетика", 3, 4),
          ("настольный теннис", 2, 0), ("шахматы", 3, 4)]),
])

`COALESCE(значение, 0)` подставляет ноль вместо `NULL` — без него
у секции без посещений в отчёте оказалась бы пустая ячейка.

### Домашнее задание 2. Средняя посещаемость

Верните одно число: средний процент посещаемости по всем записям
таблицы `посещения`, округлённый до одного знака.

In [ ]:
def средний_процент():
    return ...

In [ ]:
si.check("дз2", средний_процент, [
    ((), [(62.5,)]),
])

### Домашнее задание 3. Своя база до конца

Доведите до конца проект своей базы: три связанные таблицы, минимум
по десять записей в каждой, десять запросов и короткий текст
с выводами — что стало видно из данных. Работу принесите на урок
в виде ноутбука: ячейка со схемой, ячейка с наполнением, дальше
по ячейке на вопрос с подписанным ответом.

---

### Совет

Когда запрос выдаёт странное, разбирайте его по частям: уберите
`GROUP BY` и посмотрите на сырые строки после `JOIN`. В девяти
случаях из десяти сразу видно, что строк стало больше, чем ожидалось,
— и значит, дело в условии соединения, а не в подсчёте.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 16](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-16.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 18 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-18.ipynb)